# Kernel Optimization - make the same math run faster on a GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khanhnd61-vr/modelopt-101/blob/main/examples/05_kernel_optimization.ipynb)

**Goal.** Implement a tiled Triton matrix multiplication kernel, verify that it computes the
same result as PyTorch, and study how launch geometry and fusion change GPU performance.

This completed lab includes every experiment from `Takeaways & things to try`:

1. correctness on square, non-square, and truly ragged shapes;
2. six block configurations tuned at `2048 x 2048 x 2048`;
3. the best valid configuration benchmarked from size 512 to 4096;
4. `GROUP_M=1` versus `GROUP_M=8` as an L2-layout ablation;
5. fused MatMul + ReLU compared with PyTorch eager;
6. final correctness, latency, TFLOP/s, and speedup tables.

The fastest configuration is hardware-specific. Correctness is never optional.

**Runtime:** use `Runtime -> Change runtime type -> GPU`, then `Runtime -> Run all`.

## 1. Setup

Triton ships **inside** PyTorch, so on Colab there is nothing to `pip install`. We just
need a **GPU runtime** (`Runtime -> Change runtime type -> GPU`).

In [ ]:
import numpy as np
import torch
import triton
import triton.language as tl

GLOBAL_SEED = 0
torch.manual_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

assert torch.cuda.is_available(), "Set Runtime -> Change runtime type -> GPU"
device = torch.device("cuda")
print("device:", torch.cuda.get_device_name(0))
print("triton:", triton.__version__)

DTYPE = torch.float16
ATOL = 1e-1
RTOL = 1e-1

DEFAULT_CONFIG = {
    "name": "64x64x32",
    "BLOCK_M": 64,
    "BLOCK_N": 64,
    "BLOCK_K": 32,
    "num_warps": 4,
}

BLOCK_CONFIGS = [
    {"name": "32x32x32", "BLOCK_M": 32, "BLOCK_N": 32, "BLOCK_K": 32, "num_warps": 4},
    DEFAULT_CONFIG,
    {"name": "64x64x64", "BLOCK_M": 64, "BLOCK_N": 64, "BLOCK_K": 64, "num_warps": 4},
    {"name": "128x64x32", "BLOCK_M": 128, "BLOCK_N": 64, "BLOCK_K": 32, "num_warps": 4},
    {"name": "64x128x32", "BLOCK_M": 64, "BLOCK_N": 128, "BLOCK_K": 32, "num_warps": 4},
    {"name": "128x128x32", "BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 32, "num_warps": 8},
]

TUNE_SIZE = 2048
BENCHMARK_SIZES = [512, 1024, 2048, 4096]
GROUP_TEST_SIZES = [1024, 2048, 4096]

## 2. The operation

For `A` with shape `(M,K)` and `B` with shape `(K,N)`, matrix multiplication produces
`C=A@B` with shape `(M,N)`. The operation performs approximately `2*M*N*K` floating-point
operations. Kernel optimization changes execution, not the mathematical result.

In [ ]:
M, N, K = 1024, 1024, 1024
a = torch.randn(M, K, device=device, dtype=DTYPE)
b = torch.randn(K, N, device=device, dtype=DTYPE)

reference = torch.matmul(a, b)
print("A:", tuple(a.shape), "B:", tuple(b.shape), "C:", tuple(reference.shape))
print("FLOPs per matmul:", 2 * M * N * K)

## 3. Why tiling and grouped ordering

A Triton program computes one `BLOCK_M x BLOCK_N` output tile. It streams over `K` in
`BLOCK_K` chunks, reusing each loaded value in fast on-chip memory. The accumulator remains
FP32 until the final cast.

Grouped program ordering schedules nearby output tiles together. Those tiles reuse strips of
`A` or `B`, increasing the chance that data remains in L2 cache. Setting `GROUP_M=1` removes
this grouping benefit while leaving the arithmetic unchanged.

## 4. Tiled MatMul kernel

The kernel implements grouped PID mapping, tiled `tl.dot` accumulation, ragged-edge masking,
and an optional compile-time ReLU. The ReLU is applied to the FP32 accumulator before casting
and storing the output.

In [ ]:
@triton.jit
def matmul_kernel(
    a_ptr,
    b_ptr,
    c_ptr,
    M,
    N,
    K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    GROUP_M: tl.constexpr,
    APPLY_RELU: tl.constexpr,
):
    # Grouped one-dimensional program id -> two-dimensional output tile.
    program_id = tl.program_id(0)
    num_programs_m = tl.cdiv(M, BLOCK_M)
    num_programs_n = tl.cdiv(N, BLOCK_N)
    programs_per_group = GROUP_M * num_programs_n
    group_id = program_id // programs_per_group
    first_program_m = group_id * GROUP_M
    group_size_m = tl.minimum(num_programs_m - first_program_m, GROUP_M)
    program_m = first_program_m + (program_id % group_size_m)
    program_n = (program_id % programs_per_group) // group_size_m

    offsets_m = (program_m * BLOCK_M + tl.arange(0, BLOCK_M)) % M
    offsets_n = (program_n * BLOCK_N + tl.arange(0, BLOCK_N)) % N
    offsets_k = tl.arange(0, BLOCK_K)

    a_ptrs = (
        a_ptr
        + offsets_m[:, None] * stride_am
        + offsets_k[None, :] * stride_ak
    )
    b_ptrs = (
        b_ptr
        + offsets_k[:, None] * stride_bk
        + offsets_n[None, :] * stride_bn
    )

    accumulator = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k_block in range(0, tl.cdiv(K, BLOCK_K)):
        a_tile = tl.load(
            a_ptrs,
            mask=offsets_k[None, :] < K - k_block * BLOCK_K,
            other=0.0,
        )
        b_tile = tl.load(
            b_ptrs,
            mask=offsets_k[:, None] < K - k_block * BLOCK_K,
            other=0.0,
        )
        accumulator += tl.dot(a_tile, b_tile)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    if APPLY_RELU:
        accumulator = tl.maximum(accumulator, 0.0)

    output_m = program_m * BLOCK_M + tl.arange(0, BLOCK_M)
    output_n = program_n * BLOCK_N + tl.arange(0, BLOCK_N)
    c_ptrs = (
        c_ptr
        + output_m[:, None] * stride_cm
        + output_n[None, :] * stride_cn
    )
    output_mask = (
        (output_m[:, None] < M)
        & (output_n[None, :] < N)
    )
    tl.store(
        c_ptrs,
        accumulator.to(c_ptr.dtype.element_ty),
        mask=output_mask,
    )

## 5. Python wrapper and reusable helpers

The wrapper accepts a block configuration, grouping size, and optional fused ReLU. The helper
functions below centralize correctness and throughput calculations for every experiment.

In [ ]:
def triton_matmul(a, b, config=DEFAULT_CONFIG, group_m=8, relu=False):
    rows, inner = a.shape
    inner_b, cols = b.shape
    assert inner == inner_b, "inner dimensions must match"
    assert a.is_cuda and b.is_cuda, "inputs must be CUDA tensors"
    assert a.dtype == b.dtype, "input dtypes must match"

    output = torch.empty((rows, cols), device=a.device, dtype=a.dtype)
    grid = lambda meta: (
        triton.cdiv(rows, meta["BLOCK_M"])
        * triton.cdiv(cols, meta["BLOCK_N"]),
    )

    matmul_kernel[grid](
        a,
        b,
        output,
        rows,
        cols,
        inner,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        output.stride(0),
        output.stride(1),
        BLOCK_M=config["BLOCK_M"],
        BLOCK_N=config["BLOCK_N"],
        BLOCK_K=config["BLOCK_K"],
        GROUP_M=group_m,
        APPLY_RELU=relu,
        num_warps=config["num_warps"],
    )
    return output


def tflops(m, n, k, milliseconds):
    return (2.0 * m * n * k) / milliseconds / 1e9


def correctness_check(m, n, k, config, group_m=8, relu=False):
    torch.manual_seed(GLOBAL_SEED + m + n + k)
    left = torch.randn(m, k, device=device, dtype=DTYPE)
    right = torch.randn(k, n, device=device, dtype=DTYPE)

    expected = torch.matmul(left, right)
    if relu:
        expected = torch.relu(expected)
    actual = triton_matmul(left, right, config=config, group_m=group_m, relu=relu)

    max_error = (actual - expected).abs().max().item()
    correct = torch.allclose(actual, expected, atol=ATOL, rtol=RTOL)
    return {
        "shape": (m, n, k),
        "correct": correct,
        "max_error": max_error,
    }

## 6. Correctness first - square, non-square, and ragged

The non-square example from the original lab is retained. A second shape deliberately avoids
divisibility by the default block sizes in `M`, `N`, and `K`, so both load masks and output
masks are exercised.

In [ ]:
CORRECTNESS_SHAPES = [
    (1024, 1024, 1024),
    (512, 4096, 1024),
    (513, 4097, 1000),
]

initial_correctness = [
    correctness_check(m, n, k, DEFAULT_CONFIG)
    for m, n, k in CORRECTNESS_SHAPES
]

print(f"{'shape (M,N,K)':<24}{'correct':>10}{'max abs error':>17}")
print("-" * 51)
for row in initial_correctness:
    print(
        f'{str(row["shape"]):<24}{str(row["correct"]):>10}'
        f'{row["max_error"]:>17.5f}'
    )

assert all(row["correct"] for row in initial_correctness), (
    "The default kernel must be correct before tuning."
)

## 7. Tune six block configurations at 2048

Every configuration first passes a correctness check on the tuning matrices. Compilation or
correctness failures are recorded and excluded. The fastest valid configuration becomes the
single configuration used by the remaining benchmarks.

In [ ]:
torch.manual_seed(GLOBAL_SEED)
tune_a = torch.randn(TUNE_SIZE, TUNE_SIZE, device=device, dtype=DTYPE)
tune_b = torch.randn(TUNE_SIZE, TUNE_SIZE, device=device, dtype=DTYPE)
tune_reference = torch.matmul(tune_a, tune_b)
tune_flops = 2 * TUNE_SIZE ** 3

torch_ms_tune = triton.testing.do_bench(lambda: torch.matmul(tune_a, tune_b))
tuning_results = []

for config in BLOCK_CONFIGS:
    row = {
        "config": config,
        "name": config["name"],
        "correct": False,
        "max_error": np.nan,
        "latency_ms": np.inf,
        "tflops": 0.0,
        "speedup_vs_torch": 0.0,
        "error": "",
    }
    try:
        output = triton_matmul(tune_a, tune_b, config=config, group_m=8)
        row["max_error"] = (output - tune_reference).abs().max().item()
        row["correct"] = torch.allclose(
            output,
            tune_reference,
            atol=ATOL,
            rtol=RTOL,
        )
        if row["correct"]:
            row["latency_ms"] = triton.testing.do_bench(
                lambda cfg=config: triton_matmul(
                    tune_a,
                    tune_b,
                    config=cfg,
                    group_m=8,
                )
            )
            row["tflops"] = tune_flops / row["latency_ms"] / 1e9
            row["speedup_vs_torch"] = torch_ms_tune / row["latency_ms"]
        else:
            row["error"] = "incorrect output"
    except Exception as error:
        row["error"] = f"{type(error).__name__}: {error}"
    tuning_results.append(row)

valid_tuning_results = [
    row for row in tuning_results
    if row["correct"] and np.isfinite(row["latency_ms"])
]
if not valid_tuning_results:
    raise RuntimeError("No block configuration compiled and passed correctness.")

best_tuning_result = min(
    valid_tuning_results,
    key=lambda row: row["latency_ms"],
)
best_config = best_tuning_result["config"]

print(
    f"{'config':<14}{'correct':>10}{'max error':>13}"
    f"{'latency':>12}{'TFLOP/s':>11}{'vs torch':>11}"
)
print("-" * 71)
for row in tuning_results:
    latency_text = "failed" if not np.isfinite(row["latency_ms"]) else f'{row["latency_ms"]:.3f}ms'
    print(
        f'{row["name"]:<14}{str(row["correct"]):>10}'
        f'{row["max_error"]:>13.5f}{latency_text:>12}'
        f'{row["tflops"]:>11.1f}{row["speedup_vs_torch"]:>10.2f}x'
    )
    if row["error"]:
        print(f'  note: {row["error"]}')

print(
    f'\nSelected configuration: {best_config["name"]} '
    f'({best_tuning_result["latency_ms"]:.3f}ms, '
    f'{best_tuning_result["tflops"]:.1f} TFLOP/s)'
)

import matplotlib.pyplot as plt

valid_names = [row["name"] for row in valid_tuning_results]
x = np.arange(len(valid_names))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(
    x,
    [row["latency_ms"] for row in valid_tuning_results],
    color="#457b9d",
)
axes[0].set_xticks(x, valid_names, rotation=15)
axes[0].set_ylabel("milliseconds")
axes[0].set_title("Block tuning latency at 2048")
axes[0].grid(axis="y", alpha=0.2)

axes[1].bar(
    x,
    [row["tflops"] for row in valid_tuning_results],
    color="#2a9d8f",
)
axes[1].set_xticks(x, valid_names, rotation=15)
axes[1].set_ylabel("TFLOP/s")
axes[1].set_title("Block tuning throughput at 2048")
axes[1].grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

## 8. Recheck the selected configuration

Performance selection never overrides correctness. The winning configuration is rechecked on
all square, non-square, and ragged shapes before broader benchmarking.

In [ ]:
best_correctness = [
    correctness_check(m, n, k, best_config)
    for m, n, k in CORRECTNESS_SHAPES
]

print(f"selected config: {best_config['name']}")
for row in best_correctness:
    print(
        f"  shape={row['shape']} correct={row['correct']} "
        f"max_error={row['max_error']:.5f}"
    )

assert all(row["correct"] for row in best_correctness)

## 9. Best Triton configuration versus `torch.matmul`

The selected block configuration is benchmarked on square matrices from 512 to 4096. Both
implementations allocate an output tensor. Throughput counts the `2*N^3` MatMul FLOPs.

In [ ]:
def benchmark_matmul(size, config):
    torch.manual_seed(GLOBAL_SEED + size)
    left = torch.randn(size, size, device=device, dtype=DTYPE)
    right = torch.randn(size, size, device=device, dtype=DTYPE)

    expected = torch.matmul(left, right)
    actual = triton_matmul(left, right, config=config, group_m=8)
    correct = torch.allclose(actual, expected, atol=ATOL, rtol=RTOL)
    max_error = (actual - expected).abs().max().item()

    torch_ms = triton.testing.do_bench(lambda: torch.matmul(left, right))
    triton_ms = triton.testing.do_bench(
        lambda: triton_matmul(left, right, config=config, group_m=8)
    )

    return {
        "operation": "MatMul",
        "size": size,
        "correct": correct,
        "max_error": max_error,
        "eager_ms": torch_ms,
        "triton_ms": triton_ms,
        "eager_tflops": tflops(size, size, size, torch_ms),
        "triton_tflops": tflops(size, size, size, triton_ms),
        "speedup": torch_ms / triton_ms,
    }


matmul_results = []
for size in BENCHMARK_SIZES:
    matmul_results.append(benchmark_matmul(size, best_config))
    torch.cuda.empty_cache()

print(
    f"{'size':>6}{'correct':>10}{'max error':>12}{'torch ms':>11}"
    f"{'triton ms':>12}{'torch TF/s':>12}{'triton TF/s':>13}{'speedup':>10}"
)
print("-" * 86)
for row in matmul_results:
    print(
        f'{row["size"]:>6}{str(row["correct"]):>10}'
        f'{row["max_error"]:>12.5f}{row["eager_ms"]:>11.3f}'
        f'{row["triton_ms"]:>12.3f}{row["eager_tflops"]:>12.1f}'
        f'{row["triton_tflops"]:>13.1f}{row["speedup"]:>9.2f}x'
    )

assert all(row["correct"] for row in matmul_results)

## 10. Throughput and latency plots

These plots use the same selected Triton configuration at every size, so they show scaling
rather than per-size retuning.

In [ ]:
labels = [str(row["size"]) for row in matmul_results]
x = np.arange(len(labels))
bar_width = 0.38

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(
    x - bar_width / 2,
    [row["eager_tflops"] for row in matmul_results],
    bar_width,
    label="torch.matmul",
    color="#777777",
)
axes[0].bar(
    x + bar_width / 2,
    [row["triton_tflops"] for row in matmul_results],
    bar_width,
    label=f"Triton {best_config['name']}",
    color="#2a9d8f",
)
axes[0].set_xticks(x, labels)
axes[0].set_xlabel("square matrix size")
axes[0].set_ylabel("TFLOP/s")
axes[0].set_title("Throughput - higher is better")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.2)

axes[1].bar(
    x - bar_width / 2,
    [row["eager_ms"] for row in matmul_results],
    bar_width,
    label="torch.matmul",
    color="#777777",
)
axes[1].bar(
    x + bar_width / 2,
    [row["triton_ms"] for row in matmul_results],
    bar_width,
    label=f"Triton {best_config['name']}",
    color="#457b9d",
)
axes[1].set_xticks(x, labels)
axes[1].set_xlabel("square matrix size")
axes[1].set_ylabel("milliseconds")
axes[1].set_title("Latency - lower is better")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

## 11. Grouped-ordering ablation: `GROUP_M=1` versus `GROUP_M=8`

Both launches use the same blocks and arithmetic. Only program ordering changes. Larger
matrices are used because L2-reuse effects are most visible when the working set is large.

In [ ]:
grouping_results = []

for size in GROUP_TEST_SIZES:
    torch.manual_seed(GLOBAL_SEED + size)
    left = torch.randn(size, size, device=device, dtype=DTYPE)
    right = torch.randn(size, size, device=device, dtype=DTYPE)
    expected = torch.matmul(left, right)

    output_group_1 = triton_matmul(
        left, right, config=best_config, group_m=1
    )
    output_group_8 = triton_matmul(
        left, right, config=best_config, group_m=8
    )
    correct_1 = torch.allclose(output_group_1, expected, atol=ATOL, rtol=RTOL)
    correct_8 = torch.allclose(output_group_8, expected, atol=ATOL, rtol=RTOL)

    ms_group_1 = triton.testing.do_bench(
        lambda: triton_matmul(left, right, config=best_config, group_m=1)
    )
    ms_group_8 = triton.testing.do_bench(
        lambda: triton_matmul(left, right, config=best_config, group_m=8)
    )
    grouping_results.append({
        "size": size,
        "correct_group_1": correct_1,
        "correct_group_8": correct_8,
        "ms_group_1": ms_group_1,
        "ms_group_8": ms_group_8,
        "group_8_speedup": ms_group_1 / ms_group_8,
    })
    torch.cuda.empty_cache()

print(f"{'size':>7}{'G1 correct':>12}{'G8 correct':>12}{'G1 ms':>11}{'G8 ms':>11}{'G8/G1':>10}")
print("-" * 63)
for row in grouping_results:
    print(
        f'{row["size"]:>7}{str(row["correct_group_1"]):>12}'
        f'{str(row["correct_group_8"]):>12}{row["ms_group_1"]:>11.3f}'
        f'{row["ms_group_8"]:>11.3f}{row["group_8_speedup"]:>9.2f}x'
    )

assert all(
    row["correct_group_1"] and row["correct_group_8"]
    for row in grouping_results
)

group_labels = [str(row["size"]) for row in grouping_results]
x = np.arange(len(group_labels))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(
    x - bar_width / 2,
    [row["ms_group_1"] for row in grouping_results],
    bar_width,
    label="GROUP_M=1",
    color="#d1495b",
)
axes[0].bar(
    x + bar_width / 2,
    [row["ms_group_8"] for row in grouping_results],
    bar_width,
    label="GROUP_M=8",
    color="#2a9d8f",
)
axes[0].set_xticks(x, group_labels)
axes[0].set_xlabel("square matrix size")
axes[0].set_ylabel("milliseconds")
axes[0].set_title("Grouped-ordering latency")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.2)

axes[1].bar(
    group_labels,
    [row["group_8_speedup"] for row in grouping_results],
    color="#457b9d",
)
axes[1].axhline(1.0, color="#555555", linestyle="--")
axes[1].set_ylabel("GROUP_M=1 latency / GROUP_M=8 latency")
axes[1].set_title("Grouping benefit (>1 means GROUP_M=8 wins)")
axes[1].grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

## 12. Fused MatMul + ReLU versus PyTorch eager

Triton applies ReLU before storing the MatMul output, so global memory is written once. PyTorch
eager executes `matmul` and `relu` as separate operations. Correctness is checked against
`torch.relu(torch.matmul(A,B))`. TFLOP/s counts MatMul FLOPs only.

In [ ]:
def benchmark_fused_relu(size, config):
    torch.manual_seed(GLOBAL_SEED + size)
    left = torch.randn(size, size, device=device, dtype=DTYPE)
    right = torch.randn(size, size, device=device, dtype=DTYPE)

    expected = torch.relu(torch.matmul(left, right))
    actual = triton_matmul(
        left,
        right,
        config=config,
        group_m=8,
        relu=True,
    )
    correct = torch.allclose(actual, expected, atol=ATOL, rtol=RTOL)
    max_error = (actual - expected).abs().max().item()

    eager_ms = triton.testing.do_bench(
        lambda: torch.relu(torch.matmul(left, right))
    )
    fused_ms = triton.testing.do_bench(
        lambda: triton_matmul(
            left,
            right,
            config=config,
            group_m=8,
            relu=True,
        )
    )

    return {
        "operation": "MatMul+ReLU",
        "size": size,
        "correct": correct,
        "max_error": max_error,
        "eager_ms": eager_ms,
        "triton_ms": fused_ms,
        "eager_tflops": tflops(size, size, size, eager_ms),
        "triton_tflops": tflops(size, size, size, fused_ms),
        "speedup": eager_ms / fused_ms,
    }


fused_results = []
for size in BENCHMARK_SIZES:
    fused_results.append(benchmark_fused_relu(size, best_config))
    torch.cuda.empty_cache()

print(
    f"{'size':>6}{'correct':>10}{'max error':>12}"
    f"{'eager ms':>11}{'fused ms':>11}{'fused TF/s':>13}{'speedup':>10}"
)
print("-" * 73)
for row in fused_results:
    print(
        f'{row["size"]:>6}{str(row["correct"]):>10}'
        f'{row["max_error"]:>12.5f}{row["eager_ms"]:>11.3f}'
        f'{row["triton_ms"]:>11.3f}{row["triton_tflops"]:>13.1f}'
        f'{row["speedup"]:>9.2f}x'
    )

assert all(row["correct"] for row in fused_results)

fused_labels = [str(row["size"]) for row in fused_results]
x = np.arange(len(fused_labels))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(
    x - bar_width / 2,
    [row["eager_ms"] for row in fused_results],
    bar_width,
    label="PyTorch eager",
    color="#777777",
)
axes[0].bar(
    x + bar_width / 2,
    [row["triton_ms"] for row in fused_results],
    bar_width,
    label="Triton fused",
    color="#2a9d8f",
)
axes[0].set_xticks(x, fused_labels)
axes[0].set_xlabel("square matrix size")
axes[0].set_ylabel("milliseconds")
axes[0].set_title("Fused MatMul + ReLU latency")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.2)

axes[1].bar(
    fused_labels,
    [row["speedup"] for row in fused_results],
    color="#f4a261",
)
axes[1].axhline(1.0, color="#555555", linestyle="--")
axes[1].set_ylabel("PyTorch eager latency / Triton fused latency")
axes[1].set_title("Fusion speedup (>1 means Triton wins)")
axes[1].grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

## 13. Final correctness and performance table

The final table combines plain MatMul and fused MatMul + ReLU. Every row reports correctness,
maximum absolute error, latency, effective MatMul throughput, and speedup over the matching
PyTorch eager expression.

In [ ]:
final_results = matmul_results + fused_results

print(
    f"{'operation':<14}{'size':>7}{'correct':>10}{'max error':>12}"
    f"{'eager ms':>11}{'triton ms':>12}{'triton TF/s':>13}{'speedup':>10}"
)
print("-" * 89)
for row in final_results:
    print(
        f'{row["operation"]:<14}{row["size"]:>7}{str(row["correct"]):>10}'
        f'{row["max_error"]:>12.5f}{row["eager_ms"]:>11.3f}'
        f'{row["triton_ms"]:>12.3f}{row["triton_tflops"]:>13.1f}'
        f'{row["speedup"]:>9.2f}x'
    )

all_correct = all(row["correct"] for row in final_results)
average_matmul_speedup = np.mean([row["speedup"] for row in matmul_results])
average_grouping_effect = np.mean([
    row["group_8_speedup"] for row in grouping_results
])
average_fused_speedup = np.mean([row["speedup"] for row in fused_results])

print("\nFINAL EVALUATION")
print("=" * 80)
print(f"All square and fused benchmark rows correct: {all_correct}")
print(f"Selected block configuration: {best_config['name']}")
print(f"Average Triton MatMul speedup vs torch.matmul: {average_matmul_speedup:.2f}x")
print(f"Average GROUP_M=8 effect vs GROUP_M=1: {average_grouping_effect:.2f}x")
print(f"Average fused MatMul+ReLU speedup vs PyTorch eager: {average_fused_speedup:.2f}x")
print(
    "Block and grouping winners are GPU-specific; correctness and the measurement method "
    "are the portable conclusions."
)

## 14. Takeaways

- **Tiling:** block shape controls reuse and on-chip resource pressure. Larger is not always
  faster, so the notebook selects only among configurations that compile and pass correctness.
- **Grouped ordering:** `GROUP_M=1` and `GROUP_M=8` isolate program-layout/L2 behavior while
  keeping arithmetic and block sizes fixed.
- **Ragged shapes:** dimensions not divisible by `BLOCK_M`, `BLOCK_N`, or `BLOCK_K` validate
  both load masks and the final output-store mask.
- **Fusion:** ReLU in the accumulator avoids an additional eager operation and memory pass.
- **Benchmarking:** `triton.testing.do_bench` handles warmup and GPU synchronization; naive
  wall-clock timing is invalid for asynchronous GPU launches.
- **Hardware dependence:** cuBLAS and Triton performance vary by GPU. A speedup is an observed
  result, not a guaranteed property of one fixed block configuration.

Kernel optimization is orthogonal to quantization, distillation, pruning, and NAS. Real edge
runtimes combine compact models with specialized quantized and fused kernels.